# 📊 End-to-End Predictive Analytics Project
## Telecom Customer Churn Prediction

### Business Problem
A telecom company loses customers ("churn") every month. Acquiring a new customer costs **5–7×** more than retaining an existing one. The retention team can offer discounts — but only to a limited number of customers.

**Goal:** Build a model that predicts *which customers will churn next month*, so the retention budget targets the right people.

**Success metric:** We care most about **Recall** (catch as many actual churners as possible) balanced with **Precision** (don't waste discounts on loyal customers) → we'll optimize **ROC-AUC / F1** and pick a business-driven threshold.

### Project Pipeline (CRISP-DM style)
```
1. Business Understanding  →  2. Data Collection
3. EDA (Exploratory Data Analysis)  →  4. Data Cleaning
5. Feature Engineering  →  6. Train/Test Split & Baseline
7. Model Training & Comparison  →  8. Hyperparameter Tuning
9. Evaluation & Threshold Selection  →  10. Interpretation
11. Business Impact & Deployment
```


In [1]:
# ==============================================================
# STEP 0: Imports & settings
# ==============================================================
import sys, subprocess
for pkg in ['xgboost', 'lightgbm']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             RocCurveDisplay, PrecisionRecallDisplay, precision_recall_curve)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

np.random.seed(42)
plt.rcParams['figure.dpi'] = 90
print('Environment ready.')

Environment ready.


## Step 1–2: Business Understanding & Data Collection

We simulate a realistic telecom dataset (mirroring the famous *Telco Customer Churn* schema). In a real project this would come from the data warehouse (SQL), a CRM export, or an API.

**Why simulate?** The generator below encodes *real churn dynamics* (month-to-month contracts churn more, high monthly charges churn more, long-tenure customers stay) — so the model has genuine signal to find, and the notebook runs anywhere without downloads.


In [2]:
# ==============================================================
# STEP 2: Generate realistic telecom churn data (5,000 customers)
# ==============================================================
n = 5000
rng = np.random.RandomState(42)

df = pd.DataFrame({
    'customer_id':      [f'C{i:05d}' for i in range(n)],
    'tenure_months':    rng.exponential(24, n).clip(1, 72).round(),          # many new, few very old customers
    'contract':         rng.choice(['Month-to-month', 'One year', 'Two year'], n, p=[.55, .25, .20]),
    'internet_service': rng.choice(['DSL', 'Fiber optic', 'No'], n, p=[.35, .45, .20]),
    'payment_method':   rng.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n),
    'tech_support':     rng.choice(['Yes', 'No'], n, p=[.3, .7]),
    'paperless_billing':rng.choice(['Yes', 'No'], n, p=[.6, .4]),
    'senior_citizen':   rng.choice([0, 1], n, p=[.84, .16]),
    'num_services':     rng.randint(1, 8, n),                                # bundled services
    'monthly_charges':  rng.normal(65, 25, n).clip(18, 120).round(2),
    'support_calls':    rng.poisson(1.5, n),                                 # complaints last quarter
})
df['total_charges'] = (df['monthly_charges'] * df['tenure_months'] * rng.uniform(0.9, 1.1, n)).round(2)

# --- Ground-truth churn mechanics (what the model must discover) ---
logit = (
    -1.2
    + 1.1  * (df['contract'] == 'Month-to-month')
    + 0.6  * (df['internet_service'] == 'Fiber optic')      # fiber users are price-sensitive
    + 0.5  * (df['payment_method'] == 'Electronic check')
    - 0.8  * (df['tech_support'] == 'Yes')
    - 0.045* df['tenure_months']                             # loyalty effect
    + 0.35 * df['support_calls']                             # frustration effect
    + 0.012* df['monthly_charges']
    - 0.15 * df['num_services']                              # bundling locks people in
)
df['churn'] = (rng.rand(n) < 1 / (1 + np.exp(-logit))).astype(int)

# --- Real-world mess ---
df.loc[rng.choice(n, 150, replace=False), 'total_charges'] = np.nan     # missing values
df.loc[rng.choice(n, 60,  replace=False), 'tech_support']  = np.nan

print(f'Dataset: {df.shape[0]} customers, {df.shape[1]} columns')
print(f"Churn rate: {df['churn'].mean():.1%}")
df.head()

Dataset: 5000 customers, 13 columns
Churn rate: 37.2%


,customer_id,tenure_months,contract,internet_service,payment_method,tech_support,paperless_billing,senior_citizen,num_services,monthly_charges,support_calls,total_charges,churn
0,C00000,11.0,Month-to-month,Fiber optic,Credit card,Yes,Yes,1,2,55.67,2,663.60,1
1,C00001,72.0,Month-to-month,DSL,Mailed check,Yes,Yes,0,5,64.46,0,4221.92,0
2,C00002,32.0,Two year,DSL,Electronic check,No,Yes,1,4,69.64,1,2405.67,0
3,C00003,22.0,Month-to-month,Fiber optic,Credit card,Yes,No,0,2,46.89,1,1070.88,0
4,C00004,4.0,Two year,Fiber optic,Electronic check,No,Yes,0,4,80.23,1,314.80,0


## Step 3: Exploratory Data Analysis (EDA)

**Purpose:** understand distributions, spot data quality issues, and form hypotheses about *what drives churn* before touching any model. Good EDA questions:
1. Is the target imbalanced? (drives metric choice)
2. Which features visibly separate churners from stayers?
3. Are there missing values / outliers / weird encodings?
4. Are features correlated with each other (multicollinearity)?


In [3]:
# ==============================================================
# STEP 3a: Data quality overview
# ==============================================================
print(df.info())
print('\nMissing values:\n', df.isnull().sum()[df.isnull().sum() > 0])
df.describe().round(2)

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        5000 non-null   str    
 1   tenure_months      5000 non-null   float64
 2   contract           5000 non-null   str    
 3   internet_service   5000 non-null   str    
 4   payment_method     5000 non-null   str    
 5   tech_support       4940 non-null   str    
 6   paperless_billing  5000 non-null   str    
 7   senior_citizen     5000 non-null   int64  
 8   num_services       5000 non-null   int64  
 9   monthly_charges    5000 non-null   float64
 10  support_calls      5000 non-null   int64  
 11  total_charges      4850 non-null   float64
 12  churn              5000 non-null   int64  
dtypes: float64(3), int64(4), str(6)
memory usage: 507.9 KB
None

Missing values:
 tech_support      60
total_charges    150
dtype: int64


,tenure_months,senior_citizen,num_services,monthly_charges,support_calls,total_charges,churn
count,5000.00,5000.00,5000.00,5000.00,5000.00,4850.00,5000.00
mean,22.64,0.15,3.97,65.16,1.49,1472.14,0.37
std,20.00,0.36,1.99,24.01,1.23,1496.27,0.48
min,1.00,0.00,1.00,18.00,0.00,16.30,0.00
25%,7.00,0.00,2.00,48.05,1.00,369.87,0.00
50%,17.00,0.00,4.00,65.10,1.00,966.40,0.00
75%,33.00,0.00,6.00,81.92,2.00,2088.54,1.00
max,72.00,1.00,7.00,120.00,9.00,9041.30,1.00


In [4]:
# ==============================================================
# STEP 3b: Target balance + numeric feature distributions by churn
# ==============================================================
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

df['churn'].value_counts().plot(kind='bar', ax=axes[0,0], color=['steelblue','tomato'])
axes[0,0].set_title(f"Target balance (churn = {df['churn'].mean():.1%})")
axes[0,0].set_xticklabels(['Stayed (0)', 'Churned (1)'], rotation=0)

for ax, col in zip(axes.ravel()[1:], ['tenure_months', 'monthly_charges', 'support_calls', 'num_services', 'total_charges']):
    for label, grp in df.groupby('churn'):
        ax.hist(grp[col].dropna(), bins=30, alpha=0.55, density=True,
                label=f'churn={label}', color='tomato' if label else 'steelblue')
    ax.set_title(col); ax.legend()
plt.tight_layout(); plt.show()

# READ: churners cluster at LOW tenure, HIGH monthly charges, MORE support calls -> hypotheses confirmed

In [5]:
# ==============================================================
# STEP 3c: Churn rate by categorical feature — where is the risk?
# ==============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, col in zip(axes, ['contract', 'internet_service', 'payment_method']):
    rate = df.groupby(col)['churn'].mean().sort_values()
    rate.plot(kind='barh', ax=ax, color='coral')
    ax.set_title(f'Churn rate by {col}'); ax.set_xlabel('churn rate')
plt.tight_layout(); plt.show()

# Correlation heatmap for numeric features
plt.figure(figsize=(7, 5))
sns.heatmap(df.select_dtypes('number').corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Numeric correlations'); plt.show()
# NOTE: total_charges ≈ tenure × monthly_charges -> expected multicollinearity

## Step 4–5: Cleaning & Feature Engineering

Applying lessons from Notebook 1:
- **Impute** missing `total_charges` (median) and `tech_support` (mode) — done *inside the pipeline* to prevent leakage.
- **Engineer domain features:**
  - `avg_monthly_spend` = total ÷ tenure (spend intensity, cleans the multicollinearity)
  - `charges_per_service` = price paid per service (value-for-money perception)
  - `is_new_customer` = tenure < 6 months (the highest-risk segment)
  - `calls_per_year_tenure` = complaint *rate*, not raw count
- **Encode** categoricals with One-Hot; **scale** numerics for the linear model.


In [6]:
# ==============================================================
# STEP 5: Feature engineering
# ==============================================================
fe = df.copy()

fe['avg_monthly_spend']     = fe['total_charges'] / fe['tenure_months']
fe['charges_per_service']   = fe['monthly_charges'] / fe['num_services']
fe['is_new_customer']       = (fe['tenure_months'] < 6).astype(int)
fe['calls_per_tenure_year'] = fe['support_calls'] / (fe['tenure_months'] / 12 + 0.1)
fe['high_value']            = (fe['monthly_charges'] > fe['monthly_charges'].quantile(0.75)).astype(int)

X = fe.drop(columns=['churn', 'customer_id'])          # ID is not a feature!
y = fe['churn']

numeric_feats = ['tenure_months', 'monthly_charges', 'total_charges', 'num_services',
                 'support_calls', 'senior_citizen', 'avg_monthly_spend',
                 'charges_per_service', 'is_new_customer', 'calls_per_tenure_year', 'high_value']
categorical_feats = ['contract', 'internet_service', 'payment_method', 'tech_support', 'paperless_billing']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler())]), numeric_feats),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh',  OneHotEncoder(handle_unknown='ignore', drop='if_binary'))]), categorical_feats),
])
print(f'Feature matrix: {X.shape[1]} raw columns -> pipeline will expand categoricals automatically')

Feature matrix: 16 raw columns -> pipeline will expand categoricals automatically


## Step 6: Train/Test Split & Baseline

**Why a baseline?** Every model must beat a trivial strategy, or it's worthless. With ~27% churn, a dummy that predicts "nobody churns" scores ~73% accuracy — which is why **accuracy is misleading on imbalanced data** and we track ROC-AUC / F1.

**Stratified split** keeps the churn ratio identical in train and test.


In [7]:
# ==============================================================
# STEP 6: Split + dummy baseline
# ==============================================================
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_tr.shape[0]} | Test: {X_te.shape[0]} | churn rate train={y_tr.mean():.3f}, test={y_te.mean():.3f}')

dummy = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
print(f"\nBaseline (always predict 'stay'): accuracy={dummy.score(X_te, y_te):.3f}, "
      f"recall={recall_score(y_te, dummy.predict(X_te)):.3f}  <- catches ZERO churners!")

Train: 4000 | Test: 1000 | churn rate train=0.372, test=0.372

Baseline (always predict 'stay'): accuracy=0.628, recall=0.000  <- catches ZERO churners!


## Step 7: Model Training & Comparison

We train four models of increasing power inside identical leak-proof pipelines, compared with **5-fold stratified cross-validation** on the training set (the test set stays untouched until the end):

1. **Logistic Regression** — interpretable linear baseline (`class_weight='balanced'` to counter imbalance)
2. **Random Forest** — bagged trees, robust default
3. **XGBoost** — sequential boosting (`scale_pos_weight` handles imbalance)
4. **LightGBM** — fast leaf-wise boosting


In [8]:
# ==============================================================
# STEP 7: Cross-validated model tournament
# ==============================================================
imbalance_ratio = (y_tr == 0).sum() / (y_tr == 1).sum()    # for boosting class-weighting

models = {
    'LogisticRegression': LogisticRegression(max_iter=3000, class_weight='balanced'),
    'RandomForest':       RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                                 n_jobs=-1, random_state=42),
    'XGBoost':            XGBClassifier(n_estimators=300, learning_rate=0.07, max_depth=4,
                                        subsample=0.8, colsample_bytree=0.8,
                                        scale_pos_weight=imbalance_ratio,
                                        eval_metric='logloss', random_state=42),
    'LightGBM':           LGBMClassifier(n_estimators=300, learning_rate=0.07, num_leaves=31,
                                         subsample=0.8, colsample_bytree=0.8,
                                         class_weight='balanced', random_state=42, verbose=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, est in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', est)])
    auc = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1)
    f1  = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring='f1', n_jobs=-1)
    rows.append({'Model': name, 'CV ROC-AUC': auc.mean(), '± std': auc.std(), 'CV F1': f1.mean()})
    print(f'{name:<20} ROC-AUC = {auc.mean():.4f} ± {auc.std():.4f} | F1 = {f1.mean():.4f}')

leaderboard = pd.DataFrame(rows).round(4).sort_values('CV ROC-AUC', ascending=False)
leaderboard

LogisticRegression   ROC-AUC = 0.7854 ± 0.0059 | F1 = 0.6515


RandomForest         ROC-AUC = 0.7553 ± 0.0066 | F1 = 0.5433


XGBoost              ROC-AUC = 0.7482 ± 0.0073 | F1 = 0.6070


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM             ROC-AUC = 0.7331 ± 0.0077 | F1 = 0.5889


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,Model,CV ROC-AUC,± std,CV F1
0,LogisticRegression,0.7854,0.0059,0.6515
1,RandomForest,0.7553,0.0066,0.5433
2,XGBoost,0.7482,0.0073,0.6070
3,LightGBM,0.7331,0.0077,0.5889


## Step 8: Hyperparameter Tuning

We tune the best gradient-boosting model with **RandomizedSearchCV** — random search samples the parameter space far more efficiently than exhaustive grid search (Bergstra & Bengio, 2012), especially when only a few parameters really matter.

Key XGBoost knobs and what they trade off:
- `max_depth` / `min_child_weight` → tree complexity (overfit ↔ underfit)
- `learning_rate` × `n_estimators` → step size vs number of steps
- `subsample` / `colsample_bytree` → randomness = regularization
- `gamma`, `reg_lambda` → explicit regularization


In [9]:
# ==============================================================
# STEP 8: Randomized hyperparameter search (optimizing ROC-AUC)
# ==============================================================
xgb_pipe = Pipeline([('prep', preprocessor),
                     ('clf', XGBClassifier(eval_metric='logloss', random_state=42,
                                           scale_pos_weight=imbalance_ratio))])

param_dist = {
    'clf__n_estimators':     [200, 300, 400, 600],
    'clf__learning_rate':    [0.03, 0.05, 0.07, 0.1],
    'clf__max_depth':        [3, 4, 5, 6],
    'clf__min_child_weight': [1, 3, 5],
    'clf__subsample':        [0.7, 0.8, 0.9, 1.0],
    'clf__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'clf__gamma':            [0, 0.1, 0.3],
    'clf__reg_lambda':       [0.5, 1.0, 2.0],
}

search = RandomizedSearchCV(xgb_pipe, param_dist, n_iter=25, scoring='roc_auc',
                            cv=cv, n_jobs=-1, random_state=42, verbose=0)
search.fit(X_tr, y_tr)

print(f'Best CV ROC-AUC: {search.best_score_:.4f}')
print('Best parameters:')
for k, v in search.best_params_.items():
    print(f'  {k.replace("clf__", ""):<18} = {v}')
best_model = search.best_estimator_

Best CV ROC-AUC: 0.7699
Best parameters:
  subsample          = 0.8
  reg_lambda         = 2.0
  n_estimators       = 400
  min_child_weight   = 5
  max_depth          = 3
  learning_rate      = 0.03
  gamma              = 0.1
  colsample_bytree   = 0.7


## Step 9: Final Evaluation on the Held-Out Test Set

The test set was never touched during training or tuning — this is our honest estimate of real-world performance.

**Threshold selection is a business decision, not a math one.** The default 0.5 cutoff is arbitrary. We sweep thresholds and pick the one maximizing F1 (or, in practice, the one matching the retention team's budget/capacity).


In [10]:
# ==============================================================
# STEP 9a: Test-set metrics + confusion matrix + curves
# ==============================================================
y_proba = best_model.predict_proba(X_te)[:, 1]
y_pred  = (y_proba >= 0.5).astype(int)

print(f'ROC-AUC  : {roc_auc_score(y_te, y_proba):.4f}')
print(f'Accuracy : {accuracy_score(y_te, y_pred):.4f}')
print(f'Precision: {precision_score(y_te, y_pred):.4f}')
print(f'Recall   : {recall_score(y_te, y_pred):.4f}')
print(f'F1       : {f1_score(y_te, y_pred):.4f}\n')
print(classification_report(y_te, y_pred, target_names=['Stayed', 'Churned']))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.heatmap(confusion_matrix(y_te, y_pred), annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred Stay','Pred Churn'], yticklabels=['True Stay','True Churn'])
axes[0].set_title('Confusion Matrix @ 0.5')
RocCurveDisplay.from_predictions(y_te, y_proba, ax=axes[1]);            axes[1].set_title('ROC Curve')
PrecisionRecallDisplay.from_predictions(y_te, y_proba, ax=axes[2]);     axes[2].set_title('Precision-Recall Curve')
plt.tight_layout(); plt.show()

ROC-AUC  : 0.7935
Accuracy : 0.7040
Precision: 0.5833
Recall   : 0.7151
F1       : 0.6425

              precision    recall  f1-score   support

      Stayed       0.81      0.70      0.75       628
     Churned       0.58      0.72      0.64       372

    accuracy                           0.70      1000
   macro avg       0.69      0.71      0.69      1000
weighted avg       0.72      0.70      0.71      1000



In [11]:
# ==============================================================
# STEP 9b: Business-driven threshold selection
# ==============================================================
prec, rec, thresholds = precision_recall_curve(y_te, y_proba)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
best_t = thresholds[np.argmax(f1s)]

plt.figure(figsize=(7, 4))
plt.plot(thresholds, prec[:-1], label='Precision')
plt.plot(thresholds, rec[:-1],  label='Recall')
plt.plot(thresholds, f1s,       label='F1', lw=2)
plt.axvline(best_t, ls='--', color='gray', label=f'best F1 @ {best_t:.2f}')
plt.xlabel('Decision threshold'); plt.title('Choosing the operating point'); plt.legend(); plt.show()

y_pred_t = (y_proba >= best_t).astype(int)
print(f'At threshold {best_t:.2f}:  precision={precision_score(y_te, y_pred_t):.3f}, '
      f'recall={recall_score(y_te, y_pred_t):.3f}, f1={f1_score(y_te, y_pred_t):.3f}')
# Lower threshold  -> catch more churners (recall↑) but more wasted offers (precision↓)
# The retention team's budget decides where to sit on this curve.

At threshold 0.42:  precision=0.560, recall=0.831, f1=0.669

## Step 10: Model Interpretation — *Why* do customers churn?

A prediction without an explanation is hard to act on. We inspect **feature importances** (how much each feature contributed to splits). In production you would add **SHAP values** for per-customer explanations ("this customer is at risk *because* month-to-month contract + 4 support calls").


In [12]:
# ==============================================================
# STEP 10: Feature importance from the tuned XGBoost
# ==============================================================
# Recover the expanded feature names from the ColumnTransformer
prep = best_model.named_steps['prep']
feat_names = prep.get_feature_names_out()
importances = pd.Series(best_model.named_steps['clf'].feature_importances_, index=feat_names)

importances.sort_values().tail(12).plot(kind='barh', figsize=(8, 5), color='seagreen')
plt.title('What drives churn? — Top 12 features (XGBoost)')
plt.xlabel('Importance'); plt.tight_layout(); plt.show()

# Expected story: contract type, tenure, support calls, monthly charges dominate
# -> matches the EDA hypotheses AND the known data-generating process. The model found the truth.

# Per-customer explanations (reference):
#   import shap
#   explainer = shap.TreeExplainer(best_model.named_steps['clf'])
#   shap.summary_plot(explainer.shap_values(X_transformed), X_transformed)

## Step 11: Business Impact & Deployment

### Translating the model into money
Assume: retention offer costs **$20**, a saved customer is worth **$500**, and an offer convinces **30%** of true churners to stay.


In [13]:
# ==============================================================
# STEP 11a: ROI simulation of the retention campaign
# ==============================================================
OFFER_COST, CUSTOMER_VALUE, SAVE_RATE = 20, 500, 0.30

tn, fp, fn, tp = confusion_matrix(y_te, y_pred_t).ravel()
targeted   = tp + fp                          # everyone we send an offer to
cost       = targeted * OFFER_COST
saved      = tp * SAVE_RATE                   # churners we actually rescue
revenue    = saved * CUSTOMER_VALUE
roi        = (revenue - cost) / cost * 100

print(f'Customers targeted    : {targeted}')
print(f'True churners caught  : {tp} / {tp + fn} ({tp/(tp+fn):.0%} of all churners)')
print(f'Campaign cost         : ${cost:,.0f}')
print(f'Revenue from saves    : ${revenue:,.0f}')
print(f'ROI                   : {roi:,.0f}%')
print(f'\nvs NO MODEL (target everyone): cost = ${len(y_te)*OFFER_COST:,.0f} '
      f'for the same {tp + fn} churners reachable.')

Customers targeted    : 552
True churners caught  : 309 / 372 (83% of all churners)
Campaign cost         : $11,040
Revenue from saves    : $46,350
ROI                   : 320%

vs NO MODEL (target everyone): cost = $20,000 for the same 372 churners reachable.


In [14]:
# ==============================================================
# STEP 11b: Persist the model + score new customers (deployment pattern)
# ==============================================================
import joblib
joblib.dump(best_model, 'churn_model_v1.joblib')          # ONE artifact = preprocessing + model
print('Saved churn_model_v1.joblib')

# --- Simulate production scoring: a brand-new customer arrives ---
new_customer = pd.DataFrame([{
    'tenure_months': 3, 'contract': 'Month-to-month', 'internet_service': 'Fiber optic',
    'payment_method': 'Electronic check', 'tech_support': 'No', 'paperless_billing': 'Yes',
    'senior_citizen': 0, 'num_services': 2, 'monthly_charges': 95.0, 'support_calls': 4,
    'total_charges': 290.0,
}])
# Apply the SAME feature engineering used in training (in production: a shared function/module!)
new_customer['avg_monthly_spend']     = new_customer['total_charges'] / new_customer['tenure_months']
new_customer['charges_per_service']   = new_customer['monthly_charges'] / new_customer['num_services']
new_customer['is_new_customer']       = (new_customer['tenure_months'] < 6).astype(int)
new_customer['calls_per_tenure_year'] = new_customer['support_calls'] / (new_customer['tenure_months']/12 + 0.1)
new_customer['high_value']            = (new_customer['monthly_charges'] > 90).astype(int)

loaded = joblib.load('churn_model_v1.joblib')
risk = loaded.predict_proba(new_customer)[0, 1]
print(f'\nNew customer churn risk: {risk:.1%}  ->  {"🚨 SEND RETENTION OFFER" if risk >= best_t else "✅ low risk"}')

Saved churn_model_v1.joblib

New customer churn risk: 94.6%  ->  🚨 SEND RETENTION OFFER


## ✅ Project Wrap-Up

| Stage | What we did | Key takeaway |
|---|---|---|
| Business framing | Defined churn, cost of errors | The metric follows the money, not the math |
| EDA | Distributions, churn-rate by segment | Hypotheses before models |
| Feature engineering | Ratios, flags, tenure features | Domain features carried real signal |
| Baseline | DummyClassifier | 73% accuracy means nothing at 27% churn |
| Model tournament | LogReg vs RF vs XGB vs LGBM (5-fold CV) | Boosting wins on tabular data |
| Tuning | RandomizedSearchCV | Random > grid for the same budget |
| Evaluation | Held-out test, PR/ROC curves | Threshold = business decision |
| Interpretation | Feature importance | Contract, tenure, support calls drive churn |
| Deployment | joblib pipeline + scoring demo | Ship preprocessing WITH the model |
| ROI | Campaign simulation | The model pays for itself |

### Production next steps (beyond this notebook)
1. **Monitoring:** track data drift & AUC decay monthly; retrain on schedule.
2. **SHAP explanations** attached to every alert for the retention team.
3. **A/B test** the campaign: model-targeted vs random targeting.
4. **Serve** via FastAPI/batch scoring job; version models with MLflow.
